# Demo Training Model PBBH - Pakan Pintar Qurban

**PERINGATAN PENTING: DATASET INI SINTETIS.** Label `adg_kg_day_synthetic_label` dibuat dengan rumus simulasi untuk menguji pipeline training, bukan hasil penimbangan sapi. Model hasil notebook ini TIDAK boleh dipakai untuk rekomendasi pakan atau klaim akurasi pertumbuhan ternak sungguhan.

Tujuan notebook: demonstrasi alur upload data, preprocessing, training Random Forest Regressor, evaluasi, penyimpanan model, dan menghitung kebutuhan PBBH dari target bobot. Ganti dataset sintetis dengan catatan bobot dan pakan aktual setelah tersedia.


## 1. Instalasi dan import library

In [ ]:
!pip -q install pandas scikit-learn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


## 2. Upload dataset CSV ke Google Colab

Upload file `dataset_demo_sintetis_pbbh.csv` yang disertakan dalam paket.


In [ ]:
from google.colab import files
uploaded = files.upload()

# Sesuaikan nama file jika berbeda
file_name = 'dataset_demo_sintetis_pbbh.csv'
df = pd.read_csv(file_name)
print('Ukuran dataset:', df.shape)
df.head()


## 3. Periksa kualitas dan asal data

Pastikan kolom `data_origin` bernilai `SYNTHETIC_DEMO_NOT_FIELD_MEASUREMENT`. Jangan mencampur label sintetis ini ke data aktual.


In [ ]:
print(df['data_origin'].value_counts(dropna=False))
print(df.isna().sum())
df.describe(include='all').T


## 4. Tentukan fitur dan label

Fitur (X) berisi kondisi ternak dan jumlah empat bahan campuran yang tercatat dalam wawancara, setelah dikonversi secara asumtif menjadi kg/ekor/hari. Label (y) adalah PBBH sintetis. **Nama kolom label menegaskan bahwa ini bukan hasil pengukuran lapangan.**


In [ ]:
feature_cols = [
    'current_weight_kg',
    'age_months',
    'days_since_moved_to_current_environment',
    'rice_bran_katul_kg_day_as_fed',
    'wheat_pollard_kg_day_as_fed',
    'commercial_concentrate_nutrifeed_kg_day_as_fed',
    'tofu_dregs_kg_day_as_fed',
    'total_four_ingredient_mix_kg_day_as_fed'
]
target_col = 'adg_kg_day_synthetic_label'

X = df[feature_cols]
y = df[target_col]


## 5. Bagi data train dan test

Pembagian acak hanya dipakai untuk demo karena setiap baris dibuat sebagai simulasi independen. Untuk data nyata, jangan gunakan pembagian acak per baris jika terdapat banyak pengukuran dari sapi yang sama. Pisahkan berdasarkan `animal_id` atau kelompok ternak agar catatan satu sapi tidak bocor ke train dan test.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 6. Training model baseline Random Forest Regressor

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)


## 7. Evaluasi model

MAE dan RMSE menunjukkan selisih prediksi terhadap **label sintetis**. Nilai R² juga hanya mengukur kemampuan meniru aturan simulasi, bukan kemampuan memprediksi PBBH sapi nyata.


In [ ]:
pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print(f'MAE sintetis : {mae:.3f} kg/hari')
print(f'RMSE sintetis: {rmse:.3f} kg/hari')
print(f'R2 sintetis  : {r2:.3f}')


In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(y_test, pred, alpha=0.7)
plt.xlabel('Label PBBH sintetis (kg/hari)')
plt.ylabel('Prediksi model (kg/hari)')
plt.title('Prediksi vs label sintetis - demo saja')
plt.grid(True)
plt.show()


## 8. Contoh menghitung PBBH yang dibutuhkan untuk target bobot

Rumus ini tidak membutuhkan ML. Ini hanya menghitung laju pertambahan bobot rata-rata yang diperlukan. Nilai tersebut belum menjamin target akan tercapai.


In [ ]:
current_weight = 250.0   # kg, ubah sesuai data ternak
target_weight = 330.0    # kg, target peternak
days_remaining = 120     # hari menuju tanggal target

required_adg = (target_weight - current_weight) / days_remaining
print(f'PBBH yang dibutuhkan: {required_adg:.3f} kg/hari')


## 9. Simulasi satu prediksi (label tetap sintetis)

Isi nilai calon ransum untuk demonstrasi teknis. Jangan gunakan output ini sebagai rekomendasi jumlah pakan nyata. Model belajar dari label simulasi yang koefisiennya dibuat untuk notebook.


In [ ]:
candidate = pd.DataFrame([{
    'current_weight_kg': 250.0,
    'age_months': 18.0,
    'days_since_moved_to_current_environment': 75.0,
    'rice_bran_katul_kg_day_as_fed': 0.44,
    'wheat_pollard_kg_day_as_fed': 0.24,
    'commercial_concentrate_nutrifeed_kg_day_as_fed': 1.87,
    'tofu_dregs_kg_day_as_fed': 2.50,
    'total_four_ingredient_mix_kg_day_as_fed': 5.05
}])
predicted_demo_adg = model.predict(candidate[feature_cols])[0]
print(f'Prediksi PBBH DEMO (sintetis): {predicted_demo_adg:.3f} kg/hari')
print('Jangan gunakan angka ini untuk keputusan pemberian pakan ternak.')


## 10. Simpan model

File model hanya artefak demonstrasi. Jangan deploy sebagai model produksi untuk rekomendasi pakan sebelum dilatih dan diuji dengan data pertumbuhan aktual serta divalidasi ahli.


In [ ]:
joblib.dump({'model': model, 'feature_cols': feature_cols}, 'model_pbbh_demo_sintetis.joblib')
files.download('model_pbbh_demo_sintetis.joblib')


## 11. Cara mengganti dengan dataset lapangan

Kumpulkan satu baris untuk setiap periode pertumbuhan setiap sapi, dengan minimal:

- `animal_id`
- tanggal timbang awal dan akhir
- bobot awal dan bobot akhir (kg)
- jumlah hari antarpenimbangan
- jumlah pakan yang benar-benar dimakan, bukan hanya yang diberikan
- kadar bahan kering (BK), protein kasar (PK), dan energi/TDN dari analisis atau tabel yang tervalidasi
- bangsa, umur, jenis kelamin, kondisi awal, riwayat pindah kandang dan kesehatan

Hitung label PBBH aktual: `(bobot_akhir - bobot_awal) / jumlah_hari`. Gunakan `GroupShuffleSplit` berdasarkan `animal_id` untuk evaluasi. Hapus label sintetis dan ganti dengan PBBH aktual. Satu wawancara dengan satu peternak belum cukup untuk melatih model generalisasi.
